# MIRO AI -- Benchmark de precision, modele local (Colab, gratuit)

**Objectif** : heberger un LLM local (gpt-oss-20b par defaut) sur le GPU gratuit de Colab, l'exposer via un tunnel public (ngrok), puis pointer le pipeline MIRO AI (qui tourne sur TA machine, avec l'acces a l'instance Odoo) vers cette URL pour rejouer le benchmark adversarial et mesurer la precision reelle en conditions "locales".

**Regle de confidentialite (ne pas contourner)** : ce notebook ne sert QU'A heberger le modele. Aucune donnee client ou instance Odoo reelle ne doit transiter par cette machine Colab -- seul le corpus synthetique du projet (`document_rag/documents/`, modules Odoo de test) doit etre utilise dans le benchmark qui appellera cette URL depuis ta machine.

**Avant de commencer** :
1. Menu Colab -> `Execution` -> `Modifier le type d'execution` -> GPU (T4, tier gratuit).
2. Cree un compte gratuit sur https://ngrok.com, recupere ton authtoken (Dashboard -> Your Authtoken), colle-le dans la cellule ngrok ci-dessous.
3. Garde cet onglet ouvert et actif pendant toute la duree du benchmark (Colab gratuit se deconnecte apres ~90 min d'inactivite, session max 12h).


In [ ]:
# 1. Verifier le GPU assigne
!nvidia-smi


In [ ]:
# 2. Installer Ollama
# zstd requis par l'installeur pour l'extraction ; pciutils (lspci) requis pour la detection GPU --
# tous les deux absents de l'image Colab par defaut
!apt-get -qq update && apt-get -qq install -y zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
# 2bis. Monter Google Drive et rediriger le stockage des modeles Ollama dessus
# (evite de re-telecharger le modele a chaque nouvelle session Colab -- tier gratuit Drive = 15 Go,
# gpt-oss:20b (~13 Go) en mange la quasi-totalite, verifier l'espace disponible avant)
# Chemin range dans l'arborescence Drive existante de pmm (persistance confirmee reellement,
# cf. todo/contexte.txt Section 678 : reset complet du runtime Colab, memes fichiers/hash retrouves).
from google.colab import drive
import os

drive.mount('/content/drive')

OLLAMA_MODELS_DIR = "/content/drive/MyDrive/stagiaire@miroingenierie.com/MIRO, MHR, Integration IA Privee/ollama_models"
os.makedirs(OLLAMA_MODELS_DIR, exist_ok=True)
os.environ["OLLAMA_MODELS"] = OLLAMA_MODELS_DIR
print(f"Modeles Ollama stockes dans : {OLLAMA_MODELS_DIR}")


In [ ]:
# 3. Demarrer le serveur Ollama en arriere-plan
import subprocess, time, requests

# Tuer tout ollama serve deja lance (orphelin d'une execution precedente de cette cellule sans
# restart complet du runtime) -- sinon le nouveau process echoue au bind du port 11434 (exit 1)
# de facon silencieuse (aucune exception levee, juste un ollama_proc.poll() == 1 a verifier
# manuellement) -- cf. todo/contexte.txt Section 685/686. Rend la cellule rejouable telle quelle.
!pkill -f "ollama serve" || true
time.sleep(2)  # laisser le temps au port de se liberer avant le nouveau bind

# OLLAMA_ORIGINS=* : desactive la protection anti DNS-rebinding d'Ollama sur le header Host
# (whitelist par defaut : localhost/127.0.0.1 uniquement). Sans ca, les requetes qui arrivent
# via le tunnel ngrok (Host = <xxx>.ngrok-free.dev) sont rejetees en 403 silencieux, avant meme
# tout traitement -- cf. todo/contexte.txt Section 682 (root cause n°2, apres le fix ngrok Section 681).
os.environ["OLLAMA_ORIGINS"] = "*"

# keep_alive=-1 : ne jamais decharger le modele de la VRAM entre deux appels.
# Necessaire ici car le chargement initial depuis Google Drive (montage FUSE, cf. cellule 2bis)
# est nettement plus lent qu'un disque local -- un dechargement/rechargement en plein benchmark
# reintroduirait le meme ReadTimeout que le premier appel a froid (cf. cellule 5bis).
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
    env=os.environ,  # necessaire pour que OLLAMA_MODELS (cellule 2bis) et OLLAMA_KEEP_ALIVE soient pris en compte
)

for _ in range(30):
    try:
        r = requests.get("http://localhost:11434")
        if r.status_code == 200:
            print("Ollama pret.")
            break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(1)
else:
    raise RuntimeError("Ollama n'a pas demarre a temps -- verifier les logs du processus.")


In [ ]:
# 3bis. Verifier qu'Ollama a bien detecte le GPU (pas seulement croire le log d'install)
!ollama --version
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv


In [ ]:
# 4. Telecharger le modele candidat
# Candidat principal (voir todo/contexte.txt Section 673) : meme famille que le modele cloud actuel (GROQ_MODEL=openai/gpt-oss-20b)
MODEL = "gpt-oss:20b"

# Alternative de comparaison (si besoin, decommenter) :
# MODEL = "qwen3:14b"

!ollama pull {MODEL}


In [ ]:
# 4bis. Prechauffer le modele en VRAM avant le sanity check
# (le premier appel charge ~13 Go depuis Google Drive en VRAM -- nettement plus lent
# qu'un disque local, peut depasser largement un timeout de 60s ; voir cellule 5)
# IMPORTANT (Section 689) : le payload doit etre identique (y compris 'tools') a celui
# de la cellule 5 -- Ollama instancie un runner different selon la presence de 'tools',
# donc un prechauffage sans 'tools' ne prechauffe pas le runner tool-calling reellement
# utilise ensuite, et la cellule 5 repart d'un chargement a froid malgre OLLAMA_KEEP_ALIVE=-1.
import requests, time

_t0 = time.time()
requests.post(
    "http://localhost:11434/v1/chat/completions",
    json={
        "model": MODEL,
        "messages": [{"role": "user", "content": "hi"}],
        "tools": [{
            "type": "function",
            "function": {
                "name": "get_weather",
                "description": "Renvoie la meteo pour une ville donnee.",
                "parameters": {
                    "type": "object",
                    "properties": {"city": {"type": "string"}},
                    "required": ["city"],
                },
            },
        }],
        "stream": False,
    },
    timeout=600,
)
print(f"Modele charge en VRAM ({time.time() - _t0:.1f}s).")


In [ ]:
# 5. Sanity check local -- verifier que le tool-calling natif fonctionne
# (condition necessaire pour que le pipeline LangGraph/ReAct de MIRO AI puisse utiliser ce modele)
import requests, json as _json

test_payload = {
    "model": MODEL,
    "messages": [{"role": "user", "content": "Quel temps fait-il a Paris ?"}],
    "tools": [{
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Renvoie la meteo pour une ville donnee.",
            "parameters": {
                "type": "object",
                "properties": {"city": {"type": "string"}},
                "required": ["city"],
            },
        },
    }],
    "stream": False,
}

resp = requests.post("http://localhost:11434/v1/chat/completions", json=test_payload, timeout=300)  # large par securite, le modele est deja prechauffe (cellule precedente)
resp.raise_for_status()
data = resp.json()
tool_calls = data["choices"][0]["message"].get("tool_calls")
print("Tool calls recus :", _json.dumps(tool_calls, indent=2, ensure_ascii=False))
assert tool_calls, "Le modele n'a pas produit d'appel d'outil -- tool-calling natif KO, ne pas continuer avec ce modele."
print("OK -- tool-calling natif confirme.")


## 6. Exposer le serveur via ngrok

Avant d'executer la cellule suivante : icone cle a gauche -> ajouter un secret Colab nomme `NGROK_AUTHTOKEN` (valeur : https://dashboard.ngrok.com/get-started/your-authtoken) -> activer "Notebook access" pour ce secret.

Ne jamais coller le token en dur dans une cellule -- ce notebook est versionne dans le repo du projet.


In [ ]:
!pip install -q pyngrok
from pyngrok import ngrok, conf
from google.colab import userdata

# Secret Colab (icone cle a gauche) -- nom exact attendu : NGROK_AUTHTOKEN
# https://dashboard.ngrok.com/get-started/your-authtoken
# Ne jamais coller le token en dur ici : ce notebook est versionne dans le repo du projet.
NGROK_AUTHTOKEN = userdata.get("NGROK_AUTHTOKEN")
conf.get_default().auth_token = NGROK_AUTHTOKEN

# host_header="localhost:11434" (rewrite) : Ollama rejette en 403 silencieux toute requete
# dont le header Host ne correspond pas a localhost/127.0.0.1 (protection anti DNS-rebinding,
# verification distincte du controle CORS Origin couvert par OLLAMA_ORIGINS -- cf.
# todo/contexte.txt Section 694, root cause confirmee par isolation directe curl sans ngrok).
# Sans ce parametre, pyngrok transmet le Host <xxx>.ngrok-free.dev tel quel a Ollama, qui le rejette.
public_url = ngrok.connect(11434, "http", host_header="localhost:11434")
print("URL publique (temporaire, valable tant que ce notebook tourne) :")
print(public_url)


## 7. Cote de ta machine (ou tourne le pipeline MIRO AI / l'acces Odoo)

Dans le `.env` de `odoo-ia-prototype` (ou en variables d'environnement de la session), positionner temporairement :

```
LLM_PROVIDER=ollama
LOCAL_LLM_BASE_URL=<URL_NGROK_CI_DESSUS>/v1
LOCAL_LLM_MODEL=gpt-oss:20b
LOCAL_LLM_API_KEY=ollama
```

(`get_chat_llm()` dans `core/llm/llm_client.py` est deja cable pour ce provider -- confirme au 11/09, contrairement a la note "NotImplementedError" de `todo/plan.md` Bloc P4 Jour 2-3, qui est perimee sur ce point.)

Puis, sur ta machine :

```bash
python scripts/run_adversarial_benchmark.py
```

Le rapport sort dans `todo/verbatim_responses_sprint1.md` / `.json`, comme pour un run cloud -- comparer le taux de reussite a la derniere execution connue sur Groq.

**Ne pas oublier de remettre `LLM_PROVIDER=groq` (ou de retirer les 4 lignes ci-dessus) une fois le test termine**, pour revenir au comportement par defaut.

**Limite Colab gratuit** : session max ~12h, deconnexion apres ~90 min d'inactivite -- garder l'onglet actif et surveiller pendant le run. Si le run est trop long ou l'acces GPU instable (bascule CPU aux heures de pointe), voir Colab Pro (~12$/mois, ponctuel, A100 40 Go/L4 24 Go) -- cf. todo/contexte.txt Section 674.
